# Adapter Results Analysis
**Accuracy vs Negative Shift** scatter plots per training configuration,
with MMLU/BigBench extrinsic evaluation and learning rate robustness analysis.

In [ ]:
# ── Cell 1: Imports & Config ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from itertools import product
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "#0e1117",
    "axes.facecolor": "#161b22",
    "axes.edgecolor": "#30363d",
    "axes.labelcolor": "#c9d1d9",
    "text.color": "#c9d1d9",
    "xtick.color": "#8b949e",
    "ytick.color": "#8b949e",
    "grid.color": "#21262d",
    "grid.alpha": 0.6,
    "font.family": "monospace",
    "font.size": 11,
})

# ── Edit these ──
MODELS = ["gemma-12b", "llama-3b"]
DATASETS = ["hotpotqa", "triviaqa"]
THRESHOLDS = [0.6, 0.8]
BASE_DIR = Path("./adapters_results")

ADAPTER_COLORS = {
    "lora": "#58a6ff",
    "uiortholora": "#f78166",
    "vera": "#7ee787",
    "randlora": "#d2a8ff",
}
ADAPTER_MARKERS = {
    "lora": "o",
    "uiortholora": "s",
    "vera": "^",
    "randlora": "D",
}

In [ ]:
# ── Cell 2: Utility Functions ─────────────────────────────────────────────

def load_intrinsic(model, dataset, threshold):
    """Load per-adapter accuracy/shift CSV. Drops incomplete rows."""
    path = BASE_DIR / dataset / f"threshold_{threshold}" / f"{model}.csv"
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    df = df.dropna(subset=["accuracy"])
    df = df[df["accuracy"] > 0]
    df = df[df["tr_actual"] > 0]
    return df


def load_mmlu(dataset):
    path = BASE_DIR / dataset / "mmlu_summary.csv"
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def load_bigbench(dataset):
    path = BASE_DIR / dataset / "bigbench_summary.csv"
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def match_extrinsic(raw_adapter_name, ext_df, name_col="model_name"):
    """Fuzzy-match adapter name to a row in MMLU/BigBench summary."""
    if ext_df.empty or raw_adapter_name is None:
        return None
    col = "model_path" if "model_path" in ext_df.columns else name_col
    for _, row in ext_df.iterrows():
        val = str(row.get(col, ""))
        if raw_adapter_name in val or val.endswith(raw_adapter_name):
            return row
    return None


def parse_lr_float(lr_str):
    """Convert lr string like '5e-5' to float. Returns NaN on failure."""
    try:
        return float(lr_str)
    except (TypeError, ValueError):
        return np.nan


def enrich_with_extrinsic(df, dataset):
    """Add mmlu_acc and bigbench_acc columns to an intrinsic DataFrame."""
    mmlu = load_mmlu(dataset)
    bb = load_bigbench(dataset)

    mmlu_vals, bb_vals = [], []
    for _, r in df.iterrows():
        name = r.get("raw_adapter_name", "")

        mmlu_row = match_extrinsic(name, mmlu)
        mmlu_vals.append(mmlu_row["mmlu_acc"] if mmlu_row is not None else np.nan)

        bb_row = match_extrinsic(name, bb, name_col="model_path")
        bb_vals.append(bb_row["mean_accuracy"] if bb_row is not None else np.nan)

    df = df.copy()
    df["mmlu_acc"] = mmlu_vals
    df["bigbench_acc"] = bb_vals
    return df


def plot_scatter(model, dataset, threshold, ax):
    """Core scatter: accuracy (Y) vs negative_shift_count (X)."""
    df = load_intrinsic(model, dataset, threshold)
    if df.empty:
        ax.text(0.5, 0.5, "No data", transform=ax.transAxes,
                ha="center", va="center", fontsize=16, color="#8b949e")
        ax.set_title(f"{model} / {dataset}  (t={threshold})", fontsize=14, fontweight="bold")
        return df

    for atype in sorted(df["adapter_type"].dropna().unique()):
        sub = df[df["adapter_type"] == atype]
        color = ADAPTER_COLORS.get(atype, "#999")
        marker = ADAPTER_MARKERS.get(atype, "o")

        ax.scatter(
            sub["negative_shift_count"], sub["accuracy"],
            c=color, marker=marker, s=80, alpha=0.85,
            edgecolors="white", linewidths=0.5, label=atype, zorder=5,
        )

        if len(sub) <= 40:
            for _, row in sub.iterrows():
                parts = []
                if pd.notna(row.get("lr")):
                    parts.append(f"lr={row['lr']}")
                if pd.notna(row.get("rank")) and str(row.get("rank", "")) not in ("", "nan"):
                    parts.append(f"r={int(float(row['rank']))}")
                label = "\n".join(parts)
                if label:
                    ax.annotate(label, (row["negative_shift_count"], row["accuracy"]),
                                textcoords="offset points", xytext=(6, 6),
                                fontsize=7, color="#8b949e", alpha=0.85)

    ax.set_xlabel("Negative Shift Count  → (forgetting)", fontsize=12)
    ax.set_ylabel("Accuracy  ↑ (learning)", fontsize=12)
    ax.set_title(f"{model} / {dataset}  (threshold={threshold})", fontsize=14, fontweight="bold")
    ax.legend(loc="best", framealpha=0.3, fontsize=10)
    ax.grid(True, alpha=0.3)
    return df


def print_summary(model, dataset, threshold):
    """Print a compact summary table with optional MMLU/BigBench columns."""
    df = load_intrinsic(model, dataset, threshold)
    if df.empty:
        print(f"  No data for {model}/{dataset} t={threshold}")
        return

    df = enrich_with_extrinsic(df, dataset)
    df = df.sort_values(["adapter_type", "negative_shift_count"])

    display_cols = {
        "raw_adapter_name": "adapter",
        "adapter_type": "type",
        "lr": "lr",
        "rank": "rank",
        "accuracy": "acc",
        "negative_shift_count": "neg_shift",
        "mmlu_acc": "mmlu",
        "bigbench_acc": "bigbench",
    }
    out = df[list(display_cols.keys())].rename(columns=display_cols)
    out["acc"] = out["acc"].round(4)
    out["mmlu"] = out["mmlu"].round(4)
    out["bigbench"] = out["bigbench"].round(4)
    print(out.to_string(index=False))
    return out

In [ ]:
# ── Cell 3: Scatter — One Large Plot Per Training Config ──────────────────
THRESHOLD = 0.8

for model, dataset in product(MODELS, DATASETS):
    fig, ax = plt.subplots(figsize=(16, 10))
    df = plot_scatter(model, dataset, THRESHOLD, ax)
    plt.tight_layout()
    plt.show()

    if not df.empty:
        print(f"\n{'─'*70}")
        print(f"  {model} / {dataset}  |  threshold={THRESHOLD}  |  {len(df)} adapters")
        print(f"{'─'*70}")
        for atype in sorted(df["adapter_type"].dropna().unique()):
            sub = df[df["adapter_type"] == atype]
            print(f"  {atype:15s}  n={len(sub):3d}  "
                  f"acc=[{sub['accuracy'].min():.3f}–{sub['accuracy'].max():.3f}]  "
                  f"neg=[{sub['negative_shift_count'].min()}–{sub['negative_shift_count'].max()}]")
        print()

In [ ]:
# ── Cell 4: MMLU & BigBench — Bar Charts Per Config ───────────────────────

def plot_extrinsic_bars(model, dataset, threshold, metric, ylabel, title_suffix):
    """
    Bar chart of an extrinsic metric (mmlu_acc or bigbench_acc) per adapter,
    grouped by adapter_type, sorted by the metric value.
    """
    df = load_intrinsic(model, dataset, threshold)
    if df.empty:
        return
    df = enrich_with_extrinsic(df, dataset)
    df = df.dropna(subset=[metric])
    if df.empty:
        print(f"  No {title_suffix} matches for {model}/{dataset}")
        return

    # Deduplicate: same raw_adapter_name can appear; keep first
    df = df.drop_duplicates(subset=["raw_adapter_name"])
    df = df.sort_values(metric, ascending=True)

    fig, ax = plt.subplots(figsize=(14, max(4, len(df) * 0.45)))

    y_pos = np.arange(len(df))
    colors = [ADAPTER_COLORS.get(t, "#999") for t in df["adapter_type"]]

    bars = ax.barh(y_pos, df[metric], color=colors, edgecolor="white", linewidth=0.3, height=0.7)

    # Short labels for y-axis
    labels = []
    for _, row in df.iterrows():
        name = row["raw_adapter_name"]
        # Trim common prefixes for readability
        for prefix in ["meta-llama_Llama-3.2-3B-Instruct_", "google_gemma-3-12b-it_"]:
            name = name.replace(prefix, "")
        labels.append(name)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel(ylabel, fontsize=12)
    ax.set_title(f"{title_suffix} — {model} / {dataset}", fontsize=14, fontweight="bold")

    # Value labels on bars
    for bar, val in zip(bars, df[metric]):
        ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=8, color="#c9d1d9")

    # Add a vertical reference line for the max
    ax.axvline(df[metric].max(), color="#f0e68c", linestyle="--", alpha=0.4, linewidth=1)

    ax.grid(True, alpha=0.2, axis="x")
    plt.tight_layout()
    plt.show()


THRESHOLD = 0.8

for model, dataset in product(MODELS, DATASETS):
    plot_extrinsic_bars(model, dataset, THRESHOLD, "mmlu_acc", "MMLU Accuracy", "MMLU")
    plot_extrinsic_bars(model, dataset, THRESHOLD, "bigbench_acc", "BigBench Mean Accuracy", "BigBench")

In [ ]:
# ── Cell 5: Scatter with MMLU/BigBench as Color ──────────────────────────
def plot_scatter_colored_by_extrinsic(model, dataset, threshold, metric, metric_label):
    """
    Same accuracy-vs-forgetting scatter, but points are colored by
    an extrinsic metric (MMLU or BigBench) instead of adapter type.
    Missing extrinsic values are shown as grey hollow circles.
    """
    df = load_intrinsic(model, dataset, threshold)
    if df.empty:
        return
    df = enrich_with_extrinsic(df, dataset)

    has_metric = df[df[metric].notna()]
    no_metric = df[df[metric].isna()]

    if has_metric.empty:
        print(f"  No {metric_label} data for {model}/{dataset}")
        return

    fig, ax = plt.subplots(figsize=(16, 10))

    # Grey background points (no extrinsic match)
    if not no_metric.empty:
        ax.scatter(no_metric["negative_shift_count"], no_metric["accuracy"],
                   c="none", edgecolors="#555", s=50, alpha=0.3, linewidths=0.8,
                   label=f"no {metric_label}", zorder=3)

    # Colored points
    vmin = has_metric[metric].quantile(0.05)
    vmax = has_metric[metric].quantile(0.95)
    sc = ax.scatter(
        has_metric["negative_shift_count"], has_metric["accuracy"],
        c=has_metric[metric], cmap="RdYlGn", vmin=vmin, vmax=vmax,
        s=100, alpha=0.9, edgecolors="white", linewidths=0.5, zorder=5,
    )
    cbar = plt.colorbar(sc, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label(metric_label, fontsize=11)

    # Annotate with adapter type marker
    for _, row in has_metric.iterrows():
        atype = row.get("adapter_type", "")
        marker_char = {"lora": "L", "uiortholora": "U", "vera": "V", "randlora": "R"}.get(atype, "?")
        ax.annotate(marker_char, (row["negative_shift_count"], row["accuracy"]),
                    ha="center", va="center", fontsize=7, fontweight="bold", color="#000", zorder=6)

    ax.set_xlabel("Negative Shift Count  → (forgetting)", fontsize=12)
    ax.set_ylabel("Accuracy  ↑ (learning)", fontsize=12)
    ax.set_title(f"Acc vs Forgetting colored by {metric_label} — {model}/{dataset} (t={threshold})",
                 fontsize=13, fontweight="bold")
    ax.legend(loc="best", framealpha=0.3, fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


THRESHOLD = 0.8

for model, dataset in product(MODELS, DATASETS):
    plot_scatter_colored_by_extrinsic(model, dataset, THRESHOLD, "mmlu_acc", "MMLU Accuracy")
    plot_scatter_colored_by_extrinsic(model, dataset, THRESHOLD, "bigbench_acc", "BigBench Accuracy")

In [ ]:
# ── Cell 6: Learning Rate Robustness Analysis ────────────────────────────
def plot_lr_robustness(model, dataset, threshold):
    """
    For each adapter type, plot accuracy and neg_shift as a function of
    learning rate. This shows how sensitive each method is to LR choice.

    Two subplots:
      Left:  LR (x, log) vs Accuracy (y)
      Right: LR (x, log) vs Negative Shift Count (y)

    Each adapter type is a separate series. Error-bar-like spread if
    multiple configs share the same LR (different ranks, etc).
    """
    df = load_intrinsic(model, dataset, threshold)
    if df.empty:
        print(f"  No data for {model}/{dataset}")
        return

    df = df.copy()
    df["lr_float"] = df["lr"].apply(parse_lr_float)
    df = df.dropna(subset=["lr_float"])
    if df.empty:
        return

    adapter_types = sorted(df["adapter_type"].dropna().unique())
    if not adapter_types:
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 9))

    for atype in adapter_types:
        sub = df[df["adapter_type"] == atype].copy()
        color = ADAPTER_COLORS.get(atype, "#999")
        marker = ADAPTER_MARKERS.get(atype, "o")

        # Group by LR: compute mean and std of accuracy and neg_shift
        grouped = sub.groupby("lr_float").agg(
            acc_mean=("accuracy", "mean"),
            acc_std=("accuracy", "std"),
            acc_min=("accuracy", "min"),
            acc_max=("accuracy", "max"),
            neg_mean=("negative_shift_count", "mean"),
            neg_std=("negative_shift_count", "std"),
            neg_min=("negative_shift_count", "min"),
            neg_max=("negative_shift_count", "max"),
            count=("accuracy", "count"),
        ).reset_index().sort_values("lr_float")

        grouped["acc_std"] = grouped["acc_std"].fillna(0)
        grouped["neg_std"] = grouped["neg_std"].fillna(0)

        lrs = grouped["lr_float"].values

        # ── Accuracy vs LR ──
        ax1.plot(lrs, grouped["acc_mean"], color=color, marker=marker,
                 linewidth=2, markersize=8, label=atype, zorder=5)
        # Show range as shaded band
        if (grouped["count"] > 1).any():
            ax1.fill_between(lrs, grouped["acc_min"], grouped["acc_max"],
                             color=color, alpha=0.15, zorder=2)

        # ── Neg shift vs LR ──
        ax2.plot(lrs, grouped["neg_mean"], color=color, marker=marker,
                 linewidth=2, markersize=8, label=atype, zorder=5)
        if (grouped["count"] > 1).any():
            ax2.fill_between(lrs, grouped["neg_min"], grouped["neg_max"],
                             color=color, alpha=0.15, zorder=2)

    ax1.set_xscale("log")
    ax1.set_xlabel("Learning Rate (log scale)", fontsize=12)
    ax1.set_ylabel("Accuracy ↑", fontsize=12)
    ax1.set_title("Accuracy vs Learning Rate", fontsize=13, fontweight="bold")
    ax1.legend(loc="best", framealpha=0.3, fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(bottom=0)

    ax2.set_xscale("log")
    ax2.set_xlabel("Learning Rate (log scale)", fontsize=12)
    ax2.set_ylabel("Negative Shift Count →", fontsize=12)
    ax2.set_title("Forgetting vs Learning Rate", fontsize=13, fontweight="bold")
    ax2.legend(loc="best", framealpha=0.3, fontsize=10)
    ax2.grid(True, alpha=0.3)

    fig.suptitle(f"LR Robustness — {model} / {dataset}  (t={threshold})",
                 fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()


THRESHOLD = 0.8

for model, dataset in product(MODELS, DATASETS):
    plot_lr_robustness(model, dataset, THRESHOLD)

In [ ]:
# ── Cell 7: LR Robustness — Coefficient of Variation Table ────────────────
def lr_robustness_table(model, dataset, threshold):
    """
    For each adapter type, compute how much accuracy and forgetting vary
    across learning rates. Lower CV = more robust to LR choice.

    Metrics:
      - CV(accuracy):  std/mean across LRs  (lower = more robust)
      - Range(accuracy): max-min across LRs
      - CV(neg_shift): std/mean across LRs
      - Best acc: peak accuracy achievable
      - LR at best acc
    """
    df = load_intrinsic(model, dataset, threshold)
    if df.empty:
        return None

    df = df.copy()
    df["lr_float"] = df["lr"].apply(parse_lr_float)
    df = df.dropna(subset=["lr_float"])

    rows = []
    for atype in sorted(df["adapter_type"].dropna().unique()):
        sub = df[df["adapter_type"] == atype]
        if len(sub) < 2:
            continue

        acc_mean = sub["accuracy"].mean()
        acc_std = sub["accuracy"].std()
        acc_cv = acc_std / acc_mean if acc_mean > 0 else np.nan
        acc_range = sub["accuracy"].max() - sub["accuracy"].min()

        neg_mean = sub["negative_shift_count"].mean()
        neg_std = sub["negative_shift_count"].std()
        neg_cv = neg_std / neg_mean if neg_mean > 0 else np.nan

        best_idx = sub["accuracy"].idxmax()
        best_row = sub.loc[best_idx]

        rows.append({
            "adapter": atype,
            "n_configs": len(sub),
            "n_lrs": sub["lr_float"].nunique(),
            "acc_mean": round(acc_mean, 4),
            "acc_std": round(acc_std, 4),
            "acc_CV": round(acc_cv, 4),
            "acc_range": round(acc_range, 4),
            "neg_mean": int(neg_mean),
            "neg_CV": round(neg_cv, 4),
            "best_acc": round(best_row["accuracy"], 4),
            "best_lr": best_row["lr"],
        })

    if not rows:
        return None

    result = pd.DataFrame(rows)
    return result


THRESHOLD = 0.8

for model, dataset in product(MODELS, DATASETS):
    print(f"\n{'━'*90}")
    print(f"  LR Robustness — {model} / {dataset}  (t={THRESHOLD})")
    print(f"  Lower acc_CV = more robust to learning rate choice")
    print(f"{'━'*90}")
    tbl = lr_robustness_table(model, dataset, THRESHOLD)
    if tbl is not None:
        print(tbl.to_string(index=False))
    else:
        print("  Not enough data points")
    print()

In [ ]:
# ── Cell 8: LR Robustness — Accuracy Spread Box Plot ─────────────────────
def plot_lr_boxplot(model, dataset, threshold):
    """
    Box plot showing the distribution of accuracy across all LR settings
    for each adapter type. A tight box = robust to LR.
    """
    df = load_intrinsic(model, dataset, threshold)
    if df.empty:
        return

    adapter_types = sorted(df["adapter_type"].dropna().unique())
    if len(adapter_types) < 1:
        return

    fig, ax = plt.subplots(figsize=(12, 7))

    data_for_box = []
    labels = []
    colors = []
    for atype in adapter_types:
        sub = df[df["adapter_type"] == atype]
        if len(sub) < 2:
            continue
        data_for_box.append(sub["accuracy"].values)
        labels.append(f"{atype}\n(n={len(sub)})")
        colors.append(ADAPTER_COLORS.get(atype, "#999"))

    if not data_for_box:
        return

    bp = ax.boxplot(data_for_box, patch_artist=True, widths=0.5,
                    medianprops=dict(color="white", linewidth=2),
                    whiskerprops=dict(color="#8b949e"),
                    capprops=dict(color="#8b949e"),
                    flierprops=dict(marker="o", markerfacecolor="#f78166", markersize=5, alpha=0.6))

    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
        patch.set_edgecolor("white")
        patch.set_linewidth(0.8)

    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylabel("Accuracy", fontsize=12)
    ax.set_title(f"Accuracy Distribution Across LRs — {model}/{dataset} (t={threshold})",
                 fontsize=13, fontweight="bold")
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()


THRESHOLD = 0.8

for model, dataset in product(MODELS, DATASETS):
    plot_lr_boxplot(model, dataset, THRESHOLD)

In [ ]:
# ── Cell 9: Summary Tables with MMLU & BigBench ──────────────────────────
THRESHOLD = 0.8

for model, dataset in product(MODELS, DATASETS):
    print(f"\n{'━'*90}")
    print(f"  {model} / {dataset}  (threshold={THRESHOLD})")
    print(f"{'━'*90}")
    print_summary(model, dataset, THRESHOLD)
    print()

In [ ]:
# ── Cell 10: Threshold Comparison (0.6 vs 0.8) ───────────────────────────
for model, dataset in product(MODELS, DATASETS):
    fig, axes = plt.subplots(1, 2, figsize=(28, 10))
    for idx, t in enumerate(THRESHOLDS):
        plot_scatter(model, dataset, t, axes[idx])
    fig.suptitle(f"{model} / {dataset} — Threshold Comparison",
                 fontsize=16, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()